[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ucmoprj/RL_starcraft/blob/main/02_q_learning/walkthrough.ipynb)

# Lesson 2 walkthrough: Q-learning, one update at a time

This notebook goes with [Lesson 2](https://github.com/ucmoprj/RL_starcraft/tree/main/02_q_learning)
of *Reinforcement Learning with StarCraft II*. It needs no installation and no StarCraft:
run the cells from top to bottom with **Shift + Enter**.

We shrink the world to a corridor of 4 cells, so the Q-table has only 6 numbers and you can
follow every single change by hand.

```
[0] [1] [2] [B]
 M
```

- The marine **M** starts in cell 0. Reaching the beacon **B** gives reward **+1** and ends the episode.
- Two actions: **0 = left**, **1 = right**. Walking left in cell 0 bumps into the wall and stays put.

**Before each run, predict what you will see. Then check.**

In [ ]:
import random
import numpy as np

BEACON = 3                 # cells 0, 1, 2 are normal; cell 3 is the beacon
NAMES = ["left", "right"]


def step(s, a):
    '''The environment. Returns (next_state, reward, done).'''
    s2 = max(0, s - 1) if a == 0 else s + 1
    reward = 1.0 if s2 == BEACON else 0.0
    return s2, reward, s2 == BEACON


print(step(0, 1))   # from cell 0, walk right
print(step(2, 1))   # from cell 2, walk right -> beacon!
print(step(0, 0))   # from cell 0, walk left  -> wall, stay in cell 0

## 1. The Q-table

$Q(s, a)$ is our guess of *how much reward we will collect* if we take action $a$ in cell $s$
(and act well afterwards). One number per (cell, action). The beacon ends the episode, so it
needs no row. All guesses start at 0: we know nothing yet.

In [ ]:
def show(Q):
    print("          left    right")
    for s in range(BEACON):
        print(f"  cell {s}  {Q[s, 0]:6.3f}  {Q[s, 1]:6.3f}")


Q = np.zeros((BEACON, 2))
show(Q)

## 2. One Q-learning update

After every step $(s, a, r, s')$ we do three things (Eq. 3 and 4 in the lesson):

1. **Target** $\;y = r + \gamma \max_{a'} Q(s', a')\;$: the reward we just got plus the (discounted)
   value of the best action in the next cell. If $s'$ is the beacon there is no future, so $y = r$.
2. **Error** $\;\delta = y - Q(s, a)\;$: how wrong our guess was.
3. **Update** $\;Q(s, a) \leftarrow Q(s, a) + \alpha\,\delta\;$: move the guess a fraction $\alpha$ toward the target.

We use $\alpha = 0.5$ (learning rate) and $\gamma = 0.9$ (discount). The function below prints
all three lines so you can check the arithmetic.

In [ ]:
ALPHA, GAMMA = 0.5, 0.9


def update(Q, s, a, r, s2, done, verbose=True):
    future = 0.0 if done else Q[s2].max()
    target = r + GAMMA * future                       # Eq. 3
    error = target - Q[s, a]                          # Eq. 3
    old = Q[s, a]
    Q[s, a] += ALPHA * error                          # Eq. 4
    if verbose:
        where = "B (end)" if done else f"cell {s2}"
        print(f"cell {s} --{NAMES[a]}--> {where}, reward {r:.0f}")
        print(f"   target y = {r:.0f} + {GAMMA} * {future:.3f} = {target:.3f}")
        print(f"   error  d = {target:.3f} - {old:.3f} = {error:.3f}")
        print(f"   new Q    = {old:.3f} + {ALPHA} * {error:.3f} = {Q[s, a]:.3f}")

## 3. Stage 1: always walk right

To see the numbers clearly, the marine first walks right every time (no exploring).
Each cell below plays **one episode**: 0 → 1 → 2 → B.

**Predict:** after episode 1, which of the 6 numbers will no longer be 0?

In [ ]:
def walk_right_episode(Q):
    s, done = 0, False
    while not done:
        s2, r, done = step(s, 1)
        update(Q, s, 1, r, s2, done)
        s = s2
    print()
    show(Q)


Q = np.zeros((BEACON, 2))
walk_right_episode(Q)   # episode 1

<details><summary>What happened?</summary>

Only $Q(2, \text{right}) = 0.5$ changed. In cells 0 and 1 the reward was 0 **and** the next
cell's value was still 0, so the target was 0 and there was nothing to learn. Only the last step
received a real reward.
</details>

**Predict:** in episode 2, will $Q(1, \text{right})$ change? Why?

In [ ]:
walk_right_episode(Q)   # episode 2

<details><summary>What happened?</summary>

Yes: $Q(1, \text{right})$ became 0.225. Cell 1 still got reward 0, but its target now uses
the value of cell 2, which is 0.5: $y = 0 + 0.9 \times 0.5 = 0.45$, and half of that is 0.225.

Cell 1 learned from **a guess about cell 2**, without ever touching the beacon itself.
This is called **bootstrapping**.
</details>

In [ ]:
walk_right_episode(Q)   # episode 3

### What to notice

| | after ep. 1 | after ep. 2 | after ep. 3 |
|---|---|---|---|
| cell 2 → right | **0.500** | 0.750 | 0.875 |
| cell 1 → right | 0 | **0.225** | 0.450 |
| cell 0 → right | 0 | 0 | **0.101** |

The value flows **backwards from the beacon, one cell per episode.**

### Exercise

Without running anything: what will $Q(2, \text{right})$ be after episode 4? And $Q(1, \text{right})$?
Write your answer, then run the next cell.

<details><summary>Answer</summary>

- $Q(2,\text{right})$: $y = 1$, $\delta = 1 - 0.875 = 0.125$, new value $0.875 + 0.5 \times 0.125 = 0.9375$.
  Each episode it closes half of the remaining gap to 1.
- $Q(1,\text{right})$ is updated **before** cell 2 in the episode, so it uses the old 0.875:
  $y = 0.9 \times 0.875 = 0.7875$, $\delta = 0.7875 - 0.45 = 0.3375$, new value $0.45 + 0.169 = 0.619$.
</details>

In [ ]:
walk_right_episode(Q)   # episode 4: check your answer

**Where does it end?** Keep going and the values settle at $Q(2)=1$, $Q(1)=0.9$, $Q(0)=0.81$:
each step farther from the beacon is worth $\gamma$ times less. Run the cell to see.

In [ ]:
for _ in range(30):
    s, done = 0, False
    while not done:
        s2, r, done = step(s, 1)
        update(Q, s, 1, r, s2, done, verbose=False)
        s = s2
show(Q)

## 4. Stage 2: the agent must find the way itself

So far we *told* the marine to walk right. A real agent does not know which way is good.
It chooses with **ε-greedy** (Eq. 5):

- with probability $\varepsilon$: a **random** action (explore)
- otherwise: the action with the **highest** $Q$ in this cell (exploit); ties are broken at random

Each line below shows one episode's path. `R(greedy)` means "went right because it looked best",
`L(random)` means "went left by exploring".

**Predict:** will the first episodes be short or long? Will they get shorter?

In [ ]:
def act(Q, s, epsilon):
    if random.random() < epsilon:
        return random.randrange(2), "random"
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(random.choice(best)), ("greedy" if len(best) == 1 else "tie")


def play_episode(Q, epsilon, max_steps=100):
    s, done, path = 0, False, ["0"]
    while not done and len(path) <= max_steps:
        a, why = act(Q, s, epsilon)
        s2, r, done = step(s, a)
        update(Q, s, a, r, s2, done, verbose=False)
        path.append(f"-{NAMES[a][0].upper()}({why})-> {'B' if done else s2}")
        s = s2
    return path


random.seed(0)
Q = np.zeros((BEACON, 2))
for ep in range(1, 9):
    path = play_episode(Q, epsilon=0.3)
    print(f"Episode {ep}: {len(path) - 1} steps")
    print("  " + " ".join(path))
print()
show(Q)

<details><summary>What to notice</summary>

- At the start every $Q$ is 0, so every choice is a **tie** and the marine wanders randomly.
  Episode 2 takes many steps.
- As soon as $Q(\cdot, \text{right})$ values become positive, `greedy` picks right, and episodes shrink
  to the shortest path of 3 steps.
- Some `L(random)` steps remain: with $\varepsilon = 0.3$ the agent keeps exploring 30% of the time.
- $Q(\cdot, \text{left})$ is also learned a little (for example cell 2 → left is worth something,
  because from cell 1 you can still reach the beacon), but it stays lower than right.
</details>

## 5. Learning curve

The same thing over 50 episodes, as a plot: steps per episode.

In [ ]:
import matplotlib.pyplot as plt

random.seed(1)
Q = np.zeros((BEACON, 2))
lengths = [len(play_episode(Q, epsilon=0.3)) - 1 for _ in range(50)]

plt.plot(range(1, 51), lengths, marker="o")
plt.axhline(3, color="gray", linestyle="--", label="shortest path = 3")
plt.xlabel("episode")
plt.ylabel("steps to reach the beacon")
plt.legend()
plt.show()

## 6. Experiments

Change one thing, predict, run. Edit the cell below.

1. `epsilon=0.0`: never explore. Does it still find the beacon? (Hint: look at how ties are broken.)
2. `epsilon=1.0`: always explore. Are the episodes short? Look at the final Q-table: are the values still sensible?
3. `GAMMA = 0.5` (put `epsilon` back to 0.3). How do the final values of cells 0, 1, 2 compare?

<details><summary>Answers</summary>

1. Yes, here. With all values tied at 0 the agent picks randomly, stumbles onto the beacon, and then
   exploits. In bigger worlds pure greed often gets stuck, which is why we explore.
2. Episodes stay long (random walk), but the Q-values still approach the right values, because
   Q-learning learns about the **greedy** policy while behaving randomly (**off-policy**).
3. They drop much faster with distance: about 1, 0.5, 0.25. A small $\gamma$ makes far rewards matter less.
</details>

In [ ]:
GAMMA = 0.9          # experiment 3: try 0.5
random.seed(0)
Q = np.zeros((BEACON, 2))
lengths = [len(play_episode(Q, epsilon=0.3)) - 1 for _ in range(50)]
print("steps per episode:", lengths)
show(Q)

## Summary

- One update = **target**, **error**, **small step toward the target**.
- Values flow **backwards** from the reward, one step per visit (bootstrapping).
- **ε-greedy** balances exploring new actions and using what we learned.
- Every cell needs its own row in the table and must be **visited** to be learned.
  That is the limit that leads to Deep Q-Networks (Lesson 3).

Next: go back to [Lesson 2](https://github.com/ucmoprj/RL_starcraft/tree/main/02_q_learning)
and run the same algorithm on GridWorld and in StarCraft II.